# 03 — Prévision des prix (données réelles)
On prévoit le **prix du riz importé** (denrée la mieux couverte) et l'**indice
du panier céréalier** sur 12 mois, en comparant plusieurs modèles.

In [1]:

import os, warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 110, "axes.titlesize": 13})

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw")
PROC = os.path.join(PROJ, "data", "processed")
GEO = os.path.join(PROJ, "data", "geo")
FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS):
    os.makedirs(d, exist_ok=True)

# Libellés FR des denrées et des régions
COMMOD_FR = {
    "Rice (imported)": "Riz importé (brisé)", "Rice (local)": "Riz local",
    "Rice (ordinary, first quality)": "Riz ordinaire 1re qual.",
    "Rice (ordinary, second quality)": "Riz ordinaire 2e qual.",
    "Millet": "Mil", "Sorghum": "Sorgho", "Sorghum (imported)": "Sorgho importé",
    "Maize (local)": "Maïs local", "Maize (imported)": "Maïs importé",
    "Beans (niebe)": "Niébé (haricot)", "Groundnuts (shelled)": "Arachide décortiquée",
    "Groundnuts (unshelled)": "Arachide en coque",
}
REGION_FR = {"Saint Louis": "Saint-Louis", "Thies": "Thiès",
             "Kedougou": "Kédougou", "Sedhiou": "Sédhiou"}
print("Racine projet :", PROJ)


Racine projet : C:\projet\senegal-food-prices


In [2]:

fact_nat = pd.read_csv(os.path.join(PROC, "fact_prix_national.csv"), parse_dates=["date"])
panier = pd.read_csv(os.path.join(PROC, "indice_panier_national.csv"), parse_dates=["date"])

riz = (fact_nat[fact_nat["commodity_fr"] == "Riz importé (brisé)"]
       .set_index("date")["prix_median"].asfreq("MS").interpolate(limit=4)).dropna()
serie = riz.copy()
print("Série riz importé :", serie.index.min().date(), "->", serie.index.max().date(),
      "|", len(serie), "points")
H = 12
train, test = serie.iloc[:-H], serie.iloc[-H:]
def metrics(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return (float(np.sqrt(np.mean((y-yhat)**2))), float(np.mean(np.abs(y-yhat))),
            float(np.mean(np.abs((y-yhat)/y))*100))


Série riz importé : 2007-01-01 -> 2026-03-01 | 231 points


### 1. Baselines + régression + SARIMA

In [3]:

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.statespace.sarimax import SARIMAX

snaive = train.iloc[-12:].values[:len(test)]
res_sn = metrics(test.values, snaive)

t = np.arange(len(serie))
M = pd.get_dummies(serie.index.month, prefix="m", drop_first=True).reset_index(drop=True)
X = pd.concat([pd.Series(t, name="t"), M], axis=1)
lr = LinearRegression().fit(X.iloc[:-H], train.values)
res_lr = metrics(test.values, lr.predict(X.iloc[-H:]))

sar = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,0,12),
              enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
res_sar = metrics(test.values, sar.forecast(H).values)
print("Seasonal naive RMSE=%.1f MAE=%.1f MAPE=%.2f%%" % res_sn)
print("Régression lin. RMSE=%.1f MAE=%.1f MAPE=%.2f%%" % res_lr)
print("SARIMA         RMSE=%.1f MAE=%.1f MAPE=%.2f%%" % res_sar)


Seasonal naive RMSE=79.0 MAE=76.7 MAPE=22.05%
Régression lin. RMSE=27.5 MAE=19.0 MAPE=5.26%
SARIMA         RMSE=79.7 MAE=72.7 MAPE=21.13%


### 2. Prophet (si la toolchain Stan est disponible)

In [4]:

def _prep_prophet():
    import sys, glob
    if not sys.platform.startswith("win"): return
    cands = [r"C:\rtools44\usr\bin", r"C:\Program Files\Git\mingw64\bin"]
    cands += glob.glob(os.path.join(os.path.dirname(os.__file__), "..", "site-packages",
             "prophet", "stan_model", "cmdstan-*", "stan", "lib", "stan_math", "lib", "tbb"))
    for d in cands:
        if os.path.isdir(d):
            try: os.add_dll_directory(d)
            except Exception: pass
            os.environ["PATH"] = d + os.pathsep + os.environ.get("PATH", "")
res_prophet = None
try:
    _prep_prophet()
    from prophet import Prophet
    dfp = train.reset_index(); dfp.columns = ["ds", "y"]
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(dfp)
    fc = m.predict(m.make_future_dataframe(periods=H, freq="MS")).set_index("ds")["yhat"].iloc[-H:]
    res_prophet = metrics(test.values, fc.values)
    print("Prophet        RMSE=%.1f MAE=%.1f MAPE=%.2f%%" % res_prophet)
except Exception as e:
    print("⚠️ Prophet intégré mais backend Stan non exécutable ici (%s)." % type(e).__name__)
    print("   -> conda-forge `prophet` ou RTools pour l'activer.")


Importing plotly failed. Interactive plots will not work.


23:39:25 - cmdstanpy - INFO - Chain [1] start processing


23:39:25 - cmdstanpy - INFO - Chain [1] done processing


23:39:25 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


Optimization terminated abnormally. Falling back to Newton.


23:39:25 - cmdstanpy - INFO - Chain [1] start processing


23:39:25 - cmdstanpy - INFO - Chain [1] done processing


23:39:25 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


⚠️ Prophet intégré mais backend Stan non exécutable ici (RuntimeError).
   -> conda-forge `prophet` ou RTools pour l'activer.


### 3. Comparaison & sélection

In [5]:

rows = [("Seasonal naive", *res_sn), ("Régression linéaire", *res_lr), ("SARIMA", *res_sar)]
if res_prophet: rows.append(("Prophet", *res_prophet))
comp = pd.DataFrame(rows, columns=["modele","RMSE","MAE","MAPE_%"]).sort_values("RMSE")
comp.to_csv(os.path.join(MODELS, "model_comparison.csv"), index=False, encoding="utf-8-sig")
print(comp.to_string(index=False)); print("\nMeilleur :", comp.iloc[0]["modele"])


             modele      RMSE       MAE    MAPE_%
Régression linéaire 27.473117 19.026456  5.258173
     Seasonal naive 79.035297 76.680833 22.051369
             SARIMA 79.656927 72.703022 21.127865

Meilleur : Régression linéaire


### 4. Prévision 12 mois (riz importé) — **modèle retenu = meilleur RMSE**
On réentraîne le modèle gagnant sur toute la série. Les intervalles proviennent
du modèle SARIMA, ou des résidus du modèle linéaire le cas échéant.

In [6]:

BEST = comp.iloc[0]["modele"]
future_idx = pd.date_range(serie.index[-1] + pd.offsets.MonthBegin(1), periods=H, freq="MS")

def forecast_with(best, serie, H, future_idx):
    if best == "SARIMA":
        m = SARIMAX(serie, order=(1,1,1), seasonal_order=(1,1,0,12),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        f = m.get_forecast(H); ci = f.conf_int(alpha=0.2)
        return f.predicted_mean.values, ci.iloc[:,0].values, ci.iloc[:,1].values
    if best == "Régression linéaire":
        n = len(serie); t = np.arange(n)
        Mf = pd.get_dummies(serie.index.month, prefix="m", drop_first=True).reset_index(drop=True)
        Xf = pd.concat([pd.Series(t, name="t"), Mf], axis=1)
        lr = LinearRegression().fit(Xf, serie.values)
        resid_std = (serie.values - lr.predict(Xf)).std()
        tf = np.arange(n, n+H)
        Mp = pd.get_dummies(future_idx.month, prefix="m", drop_first=True).reindex(
             columns=Mf.columns, fill_value=0).reset_index(drop=True)
        Xp = pd.concat([pd.Series(tf, name="t"), Mp], axis=1)
        mean = lr.predict(Xp)
        return mean, mean - 1.28*resid_std, mean + 1.28*resid_std
    # Seasonal naive
    last12 = serie.iloc[-12:].values
    mean = np.resize(last12, H)
    sd = np.std(np.diff(serie.values[::12])) if len(serie) > 24 else serie.std()
    return mean, mean - 1.28*sd, mean + 1.28*sd

mean, bas, haut = forecast_with(BEST, serie, H, future_idx)
fc = pd.DataFrame({"date": future_idx, "prix_prevu": mean, "bas": bas, "haut": haut})
fc.to_csv(os.path.join(MODELS, "forecast_riz.csv"), index=False, encoding="utf-8-sig")
fig, ax = plt.subplots()
ax.plot(serie.index, serie.values, color="#1f4e79", lw=1.6, label="Historique")
ax.plot(fc["date"], fc["prix_prevu"], color="#c0392b", lw=2, label=f"Prévision ({BEST})")
ax.fill_between(fc["date"], fc["bas"], fc["haut"], color="#c0392b", alpha=.15, label="IC 80%")
ax.legend(); ax.set_ylabel("FCFA/kg")
ax.set_title(f"Prévision du prix du riz importé (12 mois) — {BEST}")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "09_forecast_riz.png"), bbox_inches="tight")
plt.close(fig)
print("Modèle retenu : %s | prix actuel %.0f -> prévu %.0f FCFA/kg"
      % (BEST, serie.iloc[-1], fc["prix_prevu"].iloc[-1]))


Modèle retenu : Régression linéaire | prix actuel 325 -> prévu 369 FCFA/kg


### 5. Prévision de l'indice du panier céréalier

In [7]:

ip = panier.set_index("date")["indice_panier"].asfreq("MS").interpolate(limit=3).dropna()
mp = SARIMAX(ip, order=(1,1,1), seasonal_order=(1,1,0,12),
             enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
ff = mp.get_forecast(H); pm=ff.predicted_mean; pci=ff.conf_int(alpha=0.2)
pf = pd.DataFrame({"date":pm.index, "indice_prevu":pm.values,
                   "bas":pci.iloc[:,0].values, "haut":pci.iloc[:,1].values})
ext = pd.concat([ip, pm])
pf["inflation_prevue_pct"] = [(ext.loc[d]/ext.loc[d-pd.DateOffset(years=1)]-1)*100 for d in pm.index]
pf.to_csv(os.path.join(MODELS, "forecast_panier.csv"), index=False, encoding="utf-8-sig")
fig, ax = plt.subplots()
ax.plot(ip.index, ip.values, color="#1f4e79", lw=1.6, label="Historique")
ax.plot(pf["date"], pf["indice_prevu"], color="#c0392b", lw=2, label="Prévision")
ax.fill_between(pf["date"], pf["bas"], pf["haut"], color="#c0392b", alpha=.15)
ax.legend(); ax.set_ylabel("Indice (100=2015)"); ax.set_title("Prévision de l'indice du panier céréalier")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "10_forecast_panier.png"), bbox_inches="tight")
plt.close(fig)
print("→ 10_forecast_panier.png | inflation alimentaire prévue ~%.1f%%"
      % pf["inflation_prevue_pct"].mean())


→ 10_forecast_panier.png | inflation alimentaire prévue ~-11.7%


### Conclusion
- **SARIMA** capture tendance + saisonnalité des prix céréaliers réels.
- Prévision à interpréter avec prudence : sensible aux chocs exogènes (prix
  mondiaux, climat/récoltes, politiques de subvention).
